# Credit Card Default Prediction

**Python · Pandas · scikit-learn · Classification**

This notebook presents a portfolio-oriented workflow adapted from academic analysis of credit-card defaults. It cleans data, compares a logistic regression baseline with a decision tree and evaluates out-of-sample results. The dataset is **not included** in this public repository.

The original coursework reported **79.75%** test accuracy for logistic regression and **69.83%** for an unpruned decision tree. This version places preprocessing inside training pipelines to prevent train/test leakage, so **results must be recalculated** and will not necessarily match the original numbers.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

DATA_FILE = Path('credit_card_defaults2.xlsx')
if not DATA_FILE.exists():
    raise FileNotFoundError('Place your locally licensed data file next to this notebook as credit_card_defaults2.xlsx')

## 1. Load and inspect the dataset

In [ ]:
df = pd.read_excel(DATA_FILE, index_col=0)
df = df.rename(columns={'PAY_0':'PAY_1', 'default payment next month':'payment_default'})
df = df.drop(columns=['ID'], errors='ignore')
assert 'payment_default' in df, 'Target column is missing; check the dataset format.'
print('Rows, columns:', df.shape)
print('Missing target values:', df['payment_default'].isna().sum())
display(df.head())

## 2. Target, features and train/test split

The original course exercise used the first 4,000 records. For comparability of scope, this notebook keeps that selection, but it applies imputation and encoding **only after the split**, within each estimator pipeline.

In [ ]:
df = df.dropna(subset=['payment_default']).iloc[:4000].copy()
df['EDUCATION'] = df['EDUCATION'].replace({0:4, 5:4, 6:4})
df['MARRIAGE'] = df['MARRIAGE'].replace({0:3})
X = df.drop(columns='payment_default')
y = df['payment_default'].astype(int)
categorical = [c for c in ['SEX','EDUCATION','MARRIAGE','PAY_1','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6'] if c in X.columns]
numerical = [c for c in X.columns if c not in categorical]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=777, stratify=y
)
print('Training:', X_train.shape, '| Test:', X_test.shape)
print('Target proportions:')
print(y_train.value_counts(normalize=True).round(3))

## 3. Leakage-aware preprocessing and models

In [ ]:
preprocess = ColumnTransformer(transformers=[
    ('numeric', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numerical),
    ('categorical', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                              ('encode', OneHotEncoder(handle_unknown='ignore'))]), categorical)
])
models = {
    'Logistic regression': LogisticRegression(max_iter=2000, random_state=777),
    'Decision tree': DecisionTreeClassifier(random_state=777)
}
results = []
fitted = {}
for name, model in models.items():
    pipe = Pipeline([('preprocess', preprocess), ('model', model)])
    pipe.fit(X_train, y_train)
    predictions = pipe.predict(X_test)
    fitted[name] = pipe
    results.append({'Model':name, 'Train accuracy':pipe.score(X_train,y_train),
                    'Test accuracy': accuracy_score(y_test,predictions)})
results_df = pd.DataFrame(results).sort_values('Test accuracy',ascending=False)
display(results_df)

## 4. Beyond accuracy

Because the outcome is imbalanced, accuracy alone is insufficient. Review class-specific precision and recall, and the confusion matrix before selecting a model for real-world credit risk use.

In [ ]:
for name, pipe in fitted.items():
    y_pred = pipe.predict(X_test)
    print('\n' + name)
    print(classification_report(y_test, y_pred, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
    plt.title(name + ' — held-out test set')
    plt.show()

## 5. Interpretation and limitations

The original assignment indicated a large train/test gap for an unpruned decision tree, suggesting overfitting. In this refined version, compare the recalculated metrics and consider tuning, cross-validation, class imbalance, fairness and business costs before proposing a production model. This is an educational analysis and should not be used for actual lending decisions.